# About SARIG
SARIG (South Australian Resources Information Gateway) is an online portal provided by the Government of South Australia. It offers access to a wide range of geoscientific data, including mineral exploration, geochemistry, and geological datasets for South Australia.

## Maximum downhole geochemistry data suite - copper

https://catalog.sarig.sa.gov.au/dataset/mesac122

In [0]:
%pip install geopandas folium gdown h3 requests
%restart_python

In [0]:
dbutils.widgets.text("catalog_name", "", "Catalog")
catalog_name = dbutils.widgets.get("catalog_name")

dbutils.widgets.text("schema_name", "", "Schema")
schema_name = dbutils.widgets.get("schema_name")

dbutils.widgets.text("volume_name", "", "Volume")
volume_name = dbutils.widgets.get("volume_name")

In [0]:
import requests

url = "https://data.sarig.sa.gov.au/Map/Download/layerDownload?layeId=1843&formatLovId=363"
file_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/copper_maxdh_GDA2020_gdb.zip"

response = requests.get(url)
with open(file_path, 'wb') as file:
    file.write(response.content)

display(file_path)

In [0]:
import zipfile

extract_path = f'/Volumes/{catalog_name}/{schema_name}/{volume_name}/copper_maxdh_GDA2020_gdb'

with zipfile.ZipFile(file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [0]:
import geopandas as gpd
import pandas as pd

gdf = gpd.read_file(f"{file_path}", driver='FileGDB')
gdf['geometry'] = gdf['geometry'].astype(str)
spark_df = spark.createDataFrame(gdf)
display(spark_df)

In [0]:
(
  spark_df.write
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(f"{catalog_name}.{schema_name}.copper_bronze")
)

In [0]:
%sql
SELECT * FROM ${catalog_name}.${schema_name}.copper_bronze LIMIT 5

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog_name}.${schema_name}.copper_silver AS 
SELECT *, st_astext(st_transform(st_setsrid(st_geomfromwkt(geometry), 7844), 4326)) as geom_4326, h3_pointash3(st_astext(st_transform(st_setsrid(st_geomfromwkt(geometry), 7844), 4326)), 9) as h3 FROM ${catalog_name}.${schema_name}.copper_bronze

In [0]:
%sql
SELECT * FROM ${catalog_name}.${schema_name}.copper_silver LIMIT 5

In [0]:
import folium
from shapely import wkt
from pyspark.sql.functions import col

# Fetch the data from the table
copper_silver_df = spark.sql(f"SELECT geom_4326, DRILLHOLE_NAME, DRILLHOLE_NO, Times_Crustal_Abundance FROM {catalog_name}.{schema_name}.copper_silver LIMIT 100")

# Convert Spark DataFrame to Pandas DataFrame
df_geom = copper_silver_df.toPandas()

# Create a folium map centered around the first point
first_geom_4326 = df_geom.iloc[0]['geom_4326']
first_point = wkt.loads(first_geom_4326)
m = folium.Map(location=[first_point.y, first_point.x], zoom_start=10)

# Add geometries to the map with tooltips and red dot icon
for _, row in df_geom.iterrows():
    folium.CircleMarker(
        location=[wkt.loads(row['geom_4326']).y, wkt.loads(row['geom_4326']).x],
        radius=row['Times_Crustal_Abundance'],  # Set radius based on Times_Crustal_Abundance
        color='red',
        fill=True,
        fill_color='red',
        tooltip=f"DRILLHOLE_NAME: {row['DRILLHOLE_NAME']}, DRILLHOLE_NO: {row['DRILLHOLE_NO']}"
    ).add_to(m)

# Display the map
display(m)

## Mineral and/or Opal Exploration Licence Applications (ELA)

https://catalog.sarig.sa.gov.au/dataset/mesac604

In [0]:
import requests

url = "https://data.sarig.sa.gov.au/Map/Download/layerDownload?layeId=1&formatLovId=20"
file_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/Exploration Licence Applications-Mineral and_or Opal_shp.zip"

response = requests.get(url)
with open(file_path, 'wb') as file:
    file.write(response.content)

display(file_path)

In [0]:
import zipfile

extract_path = f'/Volumes/{catalog_name}/{schema_name}/{volume_name}/ELA'

with zipfile.ZipFile(file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [0]:
import geopandas as gpd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from shapely import wkt
import pandas as pd

gdf_ela = gpd.read_file(f"{extract_path}/Mineral_and_or_Opal_Exploration_Licence_Applications.shp")

# Convert geometry to WKT
gdf_ela['geometry'] = gdf_ela.geometry.to_wkt()  

# Convert GeoDataFrame to Pandas DataFrame
pdf_ela = pd.DataFrame(gdf_ela)  

# Create Spark DataFrame
spark_df_ela = spark.createDataFrame(pdf_ela)

# Display the Spark DataFrame
display(spark_df_ela)

(
  spark_df_ela
  .drop("PRIOR_TEN", "RESLT_TEN", "OUTCM_DATE", "OUTCOME", "ACCPT_DATE", "OFFER_DATE", "SPEC_LOC") # remove void column (spark infers all null column as void)
  .write.mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(f"{catalog_name}.{schema_name}.ela_bronze")
)

In [0]:
spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.ela_bronze").display()

In [0]:
spark.sql(
        f"""
        CREATE OR REPLACE TABLE {catalog_name}.{schema_name}.ela_silver AS 
        SELECT *, inline(h3_tessellateaswkb(geometry, 9)) FROM {catalog_name}.{schema_name}.ela_bronze        
          """
          )

In [0]:
spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.ela_silver").display()

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog_name}.${schema_name}.copper_ela AS
SELECT copper.* except (geom_4326, EASTING_GDA2020, NORTHING_GDA2020), ela.FILE_REF, ela.APPLICANTS, ela.RECEIVED, ela.LOCATION, ela.MIN_TYPE, ela.AREA_UNIT, ela.LEGAL_AREA, ela.COMMOD_ST, ela.geometry as ela_geom, ela.cellid, ela.core, ela.chip
FROM ${catalog_name}.${schema_name}.copper_silver copper
JOIN ${catalog_name}.${schema_name}.ela_silver ela
ON ela.cellid = copper.h3
WHERE ela.core OR st_contains(st_geomfromwkb(ela.chip), st_geomfromwkt(copper.geometry));

SELECT * FROM ${catalog_name}.${schema_name}.copper_ela LIMIT 5

In [0]:
%sql
SELECT FILE_REF, APPLICANTS, COUNT(*) as cu_drillhole_count FROM ${catalog_name}.${schema_name}.copper_ela 
GROUP BY FILE_REF, APPLICANTS
ORDER BY cu_drillhole_count DESC